# Aula 05 · Delta Lake: tabelas confiáveis

## O que é
Pense em um **documento no Google Docs**: várias pessoas editam, ninguém vê um texto "pela metade" e dá para voltar a qualquer versão antiga pelo histórico.

Uma **tabela Delta** é isso para dados:

| No documento com histórico | No Delta Lake |
|---|---|
| A edição entra inteira ou não entra | **Transações ACID** |
| O histórico de versões | **Log de transações** (`_delta_log`) |
| Um modelo que não aceita campo errado | **Imposição de schema** |
| "Atualiza se existe, cria se não existe" | **MERGE** |
| "Ver como estava ontem" | **Time travel** |

## Por que importa
Quando **vários jobs leem e escrevem nas mesmas tabelas**, é o Delta Lake que evita dados corrompidos ou pela metade. Toda tabela do Databricks é Delta por padrão.

## Como aplicar
Vamos criar uma tabela de saldos de contas, atualizar com `MERGE`, consultar o histórico e voltar no tempo.

In [ ]:
%sql
CREATE SCHEMA IF NOT EXISTS growdev.nivelamento;

CREATE OR REPLACE TABLE growdev.nivelamento.saldos_delta (
  account_id STRING,
  saldo DECIMAL(14,2),
  atualizado_em TIMESTAMP
);

INSERT INTO growdev.nivelamento.saldos_delta VALUES
  ('A1', 1000.00, current_timestamp()),
  ('A2', 2500.00, current_timestamp()),
  ('A3',  300.00, current_timestamp());

SELECT * FROM growdev.nivelamento.saldos_delta ORDER BY account_id

In [ ]:
# Tentamos gravar o texto 'mil reais' na coluna saldo, que é numérica
try:
    spark.sql("INSERT INTO growdev.nivelamento.saldos_delta VALUES ('A4', 'mil reais', current_timestamp())")
except Exception as erro:
    print("O Delta recusou o dado:\n", str(erro).splitlines()[0])

### 3. MERGE: atualiza o que existe e insere o que é novo
Chegou um lote novo: a conta **A2** mudou de saldo e a conta **A4** é nova.

In [ ]:
%sql
MERGE INTO growdev.nivelamento.saldos_delta AS destino
USING (
  SELECT 'A2' AS account_id, 1800.00 AS saldo UNION ALL
  SELECT 'A4', 750.00
) AS lote_novo
ON destino.account_id = lote_novo.account_id
WHEN MATCHED THEN UPDATE SET saldo = lote_novo.saldo, atualizado_em = current_timestamp()
WHEN NOT MATCHED THEN INSERT (account_id, saldo, atualizado_em)
  VALUES (lote_novo.account_id, lote_novo.saldo, current_timestamp());

SELECT * FROM growdev.nivelamento.saldos_delta ORDER BY account_id

In [ ]:
%sql
DELETE FROM growdev.nivelamento.saldos_delta WHERE account_id = 'A1';
SELECT * FROM growdev.nivelamento.saldos_delta ORDER BY account_id

In [ ]:
%sql
DESCRIBE HISTORY growdev.nivelamento.saldos_delta

In [ ]:
# No histórico acima, procure a linha do MERGE: é a versão antes do DELETE
versao_merge = (
    spark.sql("DESCRIBE HISTORY growdev.nivelamento.saldos_delta")
    .filter("operation = 'MERGE'")
    .agg({"version": "max"})
    .first()[0]
)
print(f"Versão do MERGE: {versao_merge}")
display(spark.sql(f"SELECT * FROM growdev.nivelamento.saldos_delta VERSION AS OF {versao_merge} ORDER BY account_id"))

In [ ]:
spark.sql(f"RESTORE TABLE growdev.nivelamento.saldos_delta TO VERSION AS OF {versao_merge}")
display(spark.sql("SELECT * FROM growdev.nivelamento.saldos_delta ORDER BY account_id"))

> No projeto, a camada **silver** usa exatamente esse `MERGE` (veja `Projeto/02_silver`).

## Checklist da aula
- [ ] Criei uma tabela Delta e inseri dados
- [ ] Vi o Delta recusar um dado com tipo errado
- [ ] Atualizei e inseri com `MERGE`
- [ ] Consultei o histórico com `DESCRIBE HISTORY`
- [ ] Li uma versão antiga e restaurei a tabela